In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "8c514b99b8ab37e3cb6b59bcb625ee00dcc04957"
assert (len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH"), "Pin the reviewed pushed commit before Colab validation"
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
RUN_VERSION = "v1_panderm_base_c1_finetune"
# Operational metadata only: whichever Google account runs this notebook sets
# its own label. It is never part of the immutable experiment identity and it
# never changes the immutable experiment identity.
# A plain Run all keeps this False, so it can never take over an existing
# session by accident. Set it to True only after you have confirmed that the
# previous account's runtime is stopped; the reviewed takeover path then
# records one explicit, deterministic audit event.
MANUAL_TAKEOVER_CONFIRMED = False
assert isinstance(MANUAL_TAKEOVER_CONFIRMED, bool), "MANUAL_TAKEOVER_CONFIRMED must be a bool"
ACCOUNT_LABEL = "A"
assert ACCOUNT_LABEL in {"A", "B", "C"}, "ACCOUNT_LABEL must be A, B, or C"
ARCH = "panderm_base_vit_b16"
CHECKPOINT_FORMAT = "panderm_full_model_v1"
VARIANT = "C1"
DF_TARGET_COUNT = 585
VALIDATION_EPOCHS = 5
BATCH_SIZE = 16
ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = 128
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 10
LAYER_DECAY = 0.65
DROP_PATH = 0.2
PANDERM_UPSTREAM_REPO = "https://github.com/SiyuanYan1/PanDerm"
PANDERM_UPSTREAM_COMMIT = "fd7a80748ba7fc3e203fed88f909f4689d0d6f24"
PANDERM_CHECKPOINT_FILENAME = "panderm_bb_data6_checkpoint-499.pth"
PANDERM_CHECKPOINT_DRIVE_ID = "removed-from-public-history"
SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SHARED_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-panderm-runs")
COCA_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-coca-runs")
V1_ROOT = SHARED_RUN_ROOT / RUN_VERSION
VALIDATION_RUNS_ROOT = V1_ROOT / "validation_runs"
VALIDATION_RECORD = V1_ROOT / "validation_record.json"
LATEST_FAILURE_RECORD = V1_ROOT / "latest_validation_failure.json"
FORMAL_ROOT = V1_ROOT / "formal"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".panderm_shared_root.json"
DATA_CACHE_DIRECTORY = SHARED_RUN_ROOT / "data_cache" / "ham10000_train_val_only_v1"
CODE_DIR = Path("/content/panderm-code")
UPSTREAM_DIR = Path("/content/panderm-upstream")
WEIGHTS_CACHE = Path("/content/panderm-weights")
LOCAL_DATA_DIR = Path("/content/ham10000-data")
SMOKE_SAMPLE_DIR = Path("/content/panderm-train-smoke-sample")

# PanDerm-Base C1 full fine-tuning validation

Prerequisites: both shared Drive shortcuts must resolve physically to the reviewed roots, the user must have Editor permission, and the Colab Secret GH_TOKEN must be enabled for this notebook.

- Project shortcut: `/content/drive/MyDrive/ddpm-derm-augmentation`
- Durable run shortcut: `/content/drive/MyDrive/ddpm-derm-panderm-runs`
- A, B and C run only in sequence. The previous runtime must be stopped before another account resumes the same fixed run version.

An existing active-session marker stops normal Run all. Abrupt quota termination requires a separate explicit manual takeover after the user confirms the previous runtime is stopped. Checkpoints are saved every epoch, so the maximum expected quota loss is one incomplete epoch. No private replacement root, TTL cleanup, or automatic takeover is allowed.

Validation-only, fail-fast workflow. Formal training, test access, deployment,
and creation of a second validation attempt remain prohibited.

## Phase 0 CHECK - Drive, roots, sentinel, pinned clones, dependencies, checkpoint SHA, isolation

In [ ]:
import base64, hashlib, json, os, re, shutil, subprocess, sys, time, uuid
from google.colab import auth, drive, userdata
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project shortcut: {SHARED_PROJECT_DIR}"
assert SHARED_RUN_ROOT.is_dir(), f"missing shared run shortcut; do not create a private replacement: {SHARED_RUN_ROOT}"
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
token = userdata.get("GH_TOKEN")
assert token and len(token) > 20, "Colab Secret GH_TOKEN with read access to this repo is required (enable notebook access for accounts A/B/C)"
GH_TOKEN_PRESENT = True
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
basic_credential = base64.b64encode(("x-access-token:" + token).encode()).decode()
clone_env = os.environ.copy()
clone_env["GIT_CONFIG_COUNT"] = "1"
clone_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
clone_env["GIT_CONFIG_VALUE_0"] = "Authorization: Basic " + basic_credential
try:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True, env=clone_env)
finally:
    clone_env["GIT_CONFIG_VALUE_0"] = ""
    token = basic_credential = None
    del token, basic_credential, clone_env
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
remote = subprocess.check_output(["git", "-C", str(CODE_DIR), "remote", "get-url", "origin"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status, "clone must be a clean detached checkout of the pinned commit"
assert "@" not in remote and "x-access-token" not in remote, "clone URL must not embed a credential"
assert not UPSTREAM_DIR.exists(), f"fresh runtime required: {UPSTREAM_DIR}"
subprocess.run(["git", "clone", "--filter=blob:none", PANDERM_UPSTREAM_REPO, str(UPSTREAM_DIR)], check=True)
subprocess.run(["git", "-C", str(UPSTREAM_DIR), "checkout", "--detach", PANDERM_UPSTREAM_COMMIT], check=True)
upstream_commit = subprocess.check_output(["git", "-C", str(UPSTREAM_DIR), "rev-parse", "HEAD"], text=True).strip()
upstream_status = subprocess.check_output(["git", "-C", str(UPSTREAM_DIR), "status", "--short"], text=True).strip()
assert upstream_commit == PANDERM_UPSTREAM_COMMIT and not upstream_status, "PanDerm upstream must be a clean detached checkout of the pinned commit"
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["TORCH_HOME"] = "/content/torch-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==0.9.16", "gdown>=5.1", "pandas>=2.0", "pillow>=9.0"], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
from importlib.metadata import version
import torch
from ddpm_derm import panderm_run
assert version("timm") == "0.9.16"
assert panderm_run.UPSTREAM_COMMIT == PANDERM_UPSTREAM_COMMIT == upstream_commit
assert panderm_run.RUN_VERSION == RUN_VERSION and panderm_run.ARCH == ARCH
resolved_root = panderm_run.require_existing_shared_root(SHARED_RUN_ROOT)
# Bind every durable path to one basis. ensure_tree builds its result on the
# resolved root, so leaving the My Drive alias bound here makes relative_to fail
# on any value that came back from it. Through a Drive shortcut the two spellings
# differ, and the mismatch only surfaces once an attempt directory is created.
SHARED_RUN_ROOT = resolved_root
V1_ROOT = SHARED_RUN_ROOT / RUN_VERSION
VALIDATION_RUNS_ROOT = V1_ROOT / "validation_runs"
VALIDATION_RECORD = V1_ROOT / "validation_record.json"
LATEST_FAILURE_RECORD = V1_ROOT / "latest_validation_failure.json"
FORMAL_ROOT = V1_ROOT / "formal"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".panderm_shared_root.json"
DATA_CACHE_DIRECTORY = SHARED_RUN_ROOT / "data_cache" / "ham10000_train_val_only_v1"
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
drive_credentials, _ = google.auth.default()
DRIVE_API = build("drive", "v3", credentials=drive_credentials, cache_discovery=False)
DRIVE_PROVIDER_FIELDS = "nextPageToken,incompleteSearch,files(id,name,mimeType,parents,driveId,ownedByMe,owners(me,permissionId),resourceKey,trashed,shortcutDetails(targetId,targetMimeType,targetResourceKey))"
def drive_api_query_literal(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")
def drive_api_execute(request, resource_keys=()):
    if resource_keys:
        request.headers["X-Goog-Drive-Resource-Keys"] = panderm_run.drive_resource_key_header(tuple(resource_keys))
    return request.execute()
DRIVE_API_ABOUT = drive_api_execute(DRIVE_API.about().get(fields="user(me,permissionId)"))
# 'root' is only a files.list query alias; the API always reports the real
# My Drive root folder id in parents, so resolve that id once and use it.
DRIVE_MY_DRIVE_ROOT_ID = drive_api_execute(DRIVE_API.files().get(fileId="root", fields="id"))["id"]
assert DRIVE_MY_DRIVE_ROOT_ID and DRIVE_MY_DRIVE_ROOT_ID != "root", "the Drive API must report a resolved My Drive root folder id"
def drive_api_list_children(parent_id, name, mime_type, *, drive_id=None, parent_resource_key=None):
    query = " and ".join((f"'{drive_api_query_literal(parent_id)}' in parents", f"name = '{drive_api_query_literal(name)}'", f"mimeType = '{drive_api_query_literal(mime_type)}'", "trashed = false"))
    def fetch_page(page_token):
        kwargs = panderm_run.drive_provider_list_request_kwargs(query=query, fields=DRIVE_PROVIDER_FIELDS, drive_id=drive_id, page_token=page_token)
        request = DRIVE_API.files().list(**kwargs)
        keys = ((parent_id, parent_resource_key),) if parent_resource_key else ()
        return drive_api_execute(request, keys)
    return panderm_run.collect_drive_provider_pages(fetch_page)
shortcut_records = drive_api_list_children("root", SHARED_RUN_ROOT.name, panderm_run.DRIVE_SHORTCUT_MIME_TYPE)
# A granted account holds a shortcut; the owner account holds the folder
# itself and never a shortcut to it. Both shapes resolve to the same id.
root_folder_records = drive_api_list_children("root", SHARED_RUN_ROOT.name, panderm_run.DRIVE_FOLDER_MIME_TYPE)
DRIVE_SHARED_ROOT_TARGET = panderm_run.require_drive_shared_root_target(shortcut_records, root_folder_records, expected_alias=SHARED_RUN_ROOT.name)
DRIVE_ROOT_ID = DRIVE_SHARED_ROOT_TARGET["target_id"]
DRIVE_ROOT_RESOURCE_KEY = DRIVE_SHARED_ROOT_TARGET["target_resource_key"]
root_request = DRIVE_API.files().get(fileId=DRIVE_ROOT_ID, fields="id,name,mimeType,parents,driveId,ownedByMe,resourceKey,trashed", supportsAllDrives=True)
root_metadata = drive_api_execute(root_request, ((DRIVE_ROOT_ID, DRIVE_ROOT_RESOURCE_KEY),) if DRIVE_ROOT_RESOURCE_KEY else ())
ROOT_DRIVE_ID = root_metadata.get("driveId") or None
def wait_for_one_provider_child(parent_id, name, mime_type, phase, *, drive_id, parent_resource_key):
    started = time.monotonic()
    while True:
        records = drive_api_list_children(parent_id, name, mime_type, drive_id=drive_id, parent_resource_key=parent_resource_key)
        if len(records) == 1: return records[0]
        if len(records) > 1: raise ValueError(f"{phase}: ambiguous duplicate provider children: {records}")
        elapsed = time.monotonic() - started
        print(f"[drive-provider] {phase} visibility wait elapsed={elapsed:.1f}s", flush=True)
        if elapsed >= 30.0: raise FileNotFoundError(f"{phase}: provider child was not visible after 30 seconds")
        time.sleep(5.0)
api_mount_probe_name = f".panderm_api_mount_identity_{uuid.uuid4().hex}.json"
api_mount_probe_path = Path("/content/drive/MyDrive") / api_mount_probe_name
try:
    api_mount_probe_path.write_text('{"probe":"drive-api-fuse-account"}\n', encoding="utf-8")
    api_mount_probe_metadata = wait_for_one_provider_child("root", api_mount_probe_name, panderm_run.DRIVE_JSON_MIME_TYPE, "Phase 0 Drive API/FUSE account probe", drive_id=None, parent_resource_key=None)
    DRIVE_API_FUSE_ACCOUNT_ALIGNMENT = panderm_run.require_drive_api_fuse_account_alignment(api_mount_probe_metadata, expected_probe_name=api_mount_probe_name, expected_root_id=DRIVE_MY_DRIVE_ROOT_ID)
finally:
    api_mount_probe_path.unlink(missing_ok=True)
topology_probe_name = f".panderm_lock_topology_probe_{uuid.uuid4().hex}.json"
topology_probe_path = SHARED_RUN_ROOT / topology_probe_name
try:
    topology_probe_path.write_text('{"probe":"validation-lock-topology"}\n', encoding="utf-8")
    topology_probe_metadata = wait_for_one_provider_child(DRIVE_ROOT_ID, topology_probe_name, panderm_run.DRIVE_JSON_MIME_TYPE, "Phase 0 lock topology probe", drive_id=ROOT_DRIVE_ID, parent_resource_key=DRIVE_ROOT_RESOURCE_KEY)
    VALIDATION_LOCK_STORAGE_TOPOLOGY = panderm_run.require_validation_lock_storage_topology(root_metadata, topology_probe_metadata, expected_root_id=DRIVE_ROOT_ID, expected_probe_name=topology_probe_name)
finally:
    topology_probe_path.unlink(missing_ok=True)
phase0_version_records = drive_api_list_children(DRIVE_ROOT_ID, RUN_VERSION, panderm_run.DRIVE_FOLDER_MIME_TYPE, drive_id=ROOT_DRIVE_ID, parent_resource_key=DRIVE_ROOT_RESOURCE_KEY)
PHASE0_VERSION_PROVIDER = panderm_run.require_validation_version_provider_state(phase0_version_records, expected_parent_id=DRIVE_ROOT_ID, run_version=RUN_VERSION, topology=VALIDATION_LOCK_STORAGE_TOPOLOGY, local_version_exists=V1_ROOT.exists())
if PHASE0_VERSION_PROVIDER is not None:
    phase0_version_resource_key = PHASE0_VERSION_PROVIDER.get("resourceKey")
    phase0_version_resource_key = phase0_version_resource_key or ""
    phase0_active_records = drive_api_list_children(PHASE0_VERSION_PROVIDER["id"], panderm_run.ACTIVE_SESSION_FILENAME, panderm_run.DRIVE_JSON_MIME_TYPE, drive_id=ROOT_DRIVE_ID, parent_resource_key=phase0_version_resource_key)
    if phase0_active_records:
        panderm_run.require_active_session_provider_state(phase0_active_records, expected_parent_id=PHASE0_VERSION_PROVIDER["id"], topology=VALIDATION_LOCK_STORAGE_TOPOLOGY, expected_present=True)
        assert MANUAL_TAKEOVER_CONFIRMED is True, "active session exists; confirm the previous runtime is stopped and use the reviewed manual-takeover path"
    else:
        panderm_run.require_active_session_provider_state([], expected_parent_id=PHASE0_VERSION_PROVIDER["id"], topology=VALIDATION_LOCK_STORAGE_TOPOLOGY, expected_present=False)
drive_probe = panderm_run.probe_shared_drive(SHARED_RUN_ROOT)
assert SHARED_ROOT_SENTINEL.is_file(), f"missing shared-root sentinel; do not recreate it: {SHARED_ROOT_SENTINEL}"
sentinel = panderm_run.create_or_validate_sentinel(SHARED_ROOT_SENTINEL, shortcut_alias="ddpm-derm-panderm-runs", resolved_path=str(resolved_root), drive_folder_id=DRIVE_ROOT_ID, run_version=RUN_VERSION)
assert sentinel.get("drive_folder_id") in (None, DRIVE_ROOT_ID), "shared-root sentinel Drive folder id drift"
panderm_run.require_shared_root_sentinel_identity(sentinel, shortcut_alias="ddpm-derm-panderm-runs", resolved_root=resolved_root, run_version=RUN_VERSION)
DURABLE_ROOT_PROVIDER_IDENTITY = panderm_run.build_durable_root_provider_identity(root_metadata, expected_root_id=DRIVE_ROOT_ID, shortcut_target_resource_key=DRIVE_ROOT_RESOURCE_KEY, shared_root_uuid=sentinel["shared_root_uuid"], topology=VALIDATION_LOCK_STORAGE_TOPOLOGY)
assert SHARED_RUN_ROOT != COCA_RUN_ROOT and COCA_RUN_ROOT not in SHARED_RUN_ROOT.parents, "PanDerm output root must be fully isolated from the CoCa runs"
coca_guard_paths = sorted(COCA_RUN_ROOT.rglob("validation_record.json")) + sorted(COCA_RUN_ROOT.rglob("_COMPLETED.json")) if COCA_RUN_ROOT.is_dir() else []
project_guard_paths = [SHARED_PROJECT_DIR / name for name in ("README.md", "HANDOFF.md", "EXPERIMENT_LOG.md")]
existing_attempts = sorted(path for path in VALIDATION_RUNS_ROOT.iterdir() if path.is_dir()) if VALIDATION_RUNS_ROOT.is_dir() else []
assert len(existing_attempts) <= 1, f"only one validation attempt directory is permitted: {existing_attempts}"
attempt_guard_paths = sorted(path for attempt in existing_attempts for path in attempt.rglob("*") if path.is_file())
# Durable checkpoints and results of the SAME fixed run version are expected
# when the next account resumes it, so they are neither rejected here nor
# treated as immutable guard files. Only completion/formal markers still
# block starting work on this attempt.
resumable_gate_roots = [attempt / "non_collapse_gate" for attempt in existing_attempts]
resumable_attempt_artifacts = [path for path in attempt_guard_paths if any(path.is_relative_to(root) for root in resumable_gate_roots)]
unexpected_formal_names = {"_COMPLETED.json", "validation_record.json"}
unexpected_attempt_artifacts = [path for path in attempt_guard_paths if path.name in unexpected_formal_names]
assert not unexpected_attempt_artifacts, f"unexpected completed/formal artifacts in the unique attempt; inspect manually: {unexpected_attempt_artifacts}"
guard_paths = [path for path in coca_guard_paths + project_guard_paths + attempt_guard_paths if path.is_file() and path not in resumable_attempt_artifacts]
before_guard = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
panderm_run.require_no_deployment_contamination(CODE_DIR)
assert not V1_ROOT.is_symlink(), f"validation version parent must not be a symlink: {V1_ROOT}"
if V1_ROOT.exists():
    assert V1_ROOT.is_dir(), f"validation version parent must be a directory: {V1_ROOT}"
    preexisting_v1_root = V1_ROOT.resolve(strict=True)
    assert preexisting_v1_root.is_relative_to(resolved_root) and preexisting_v1_root.parent.samefile(resolved_root), "validation version parent escaped the shared root"
    assert not VALIDATION_RECORD.exists() and not LATEST_FAILURE_RECORD.exists() and not FORMAL_ROOT.exists(), "existing validation/formal state blocks a fresh run"
VALIDATION_SESSION = panderm_run.session_marker(ACCOUNT_LABEL, "resume", {"run_version": RUN_VERSION})
VALIDATION_SESSION_ID = VALIDATION_SESSION["session_id"]
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "CUDA is required after shared-root and active-session preflight"
print(json.dumps({"commit": commit, "upstream_commit": upstream_commit, "drive_probe": drive_probe, "shared_root_source": DRIVE_SHARED_ROOT_TARGET["source"], "storage_topology": VALIDATION_LOCK_STORAGE_TOPOLOGY, "manual_takeover_confirmed": MANUAL_TAKEOVER_CONFIRMED, "guard_files": len(guard_paths), "existing_attempts": [path.name for path in existing_attempts]}, indent=2))

### Phase 0 CHECK - official checkpoint download and pinned SHA-256

In [ ]:
WEIGHTS_CACHE.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = WEIGHTS_CACHE / PANDERM_CHECKPOINT_FILENAME
if not CHECKPOINT_PATH.is_file():
    import gdown
    gdown.download(id=PANDERM_CHECKPOINT_DRIVE_ID, output=str(CHECKPOINT_PATH), quiet=False)
assert CHECKPOINT_PATH.is_file(), f"PanDerm checkpoint download failed: {CHECKPOINT_PATH}"
observed_checkpoint_sha256 = sha256(CHECKPOINT_PATH)
print("checkpoint bytes:", CHECKPOINT_PATH.stat().st_size)
print("checkpoint sha256:", observed_checkpoint_sha256)
print("expected (pinned):", panderm_run.EXPECTED_CHECKPOINT_SHA256)
checkpoint_sha256 = panderm_run.require_checkpoint_sha256(CHECKPOINT_PATH)
provenance = panderm_run.summarize_provenance()
assert provenance["license_review"]["license"] == "CC-BY-NC-ND-4.0"
assert provenance["license_review"]["finetuning_allowed"] is True
assert provenance["license_review"]["deployment_allowed"] is False
assert provenance["license_review"]["sharing_adapted_weights_allowed"] is False
assert provenance["contamination_review"]["image_level_ham10000_overlap"] == "not_independently_excludable"
assert provenance["contamination_review"]["independent_audit_possible"] is False
assert provenance["contamination_review"]["exact_fixed_validation_test_overlap"] == "unproven"
assert provenance["contamination_review"]["patient_level_overlap"] == "not_excludable"
assert provenance["contamination_review"]["ham10000_in_upstream_finetuning_or_evaluation"] == "yes_evaluation_benchmark"
assert provenance["contamination_review"]["loaded_checkpoint_is_pretraining_only"] is True
assert provenance["claim_boundary"] == "suggestive_exploratory_only"
assert provenance["deployment_allowed"] is False
provenance_clearance = panderm_run.require_provenance_clearance(upstream_commit=upstream_commit, checkpoint_sha256=checkpoint_sha256, purpose=panderm_run.VALIDATION_ONLY)
print(json.dumps(provenance_clearance, indent=2))

## Phase 1 CHECK - checkpoint layout, real CPU model load, primitive run identity and dependency versions

In [ ]:
import gc
assert "DDPM_DERM_DATA_DIR" not in os.environ, "Phase 1 must not bind a training dataset"
from ddpm_derm import panderm
phase1_started = time.monotonic()
print("[Phase 1] START checkpoint/model/identity checks", flush=True)
phase1_factory = panderm.load_upstream_model_factory(UPSTREAM_DIR)
pretrained_state = panderm.load_pretrained_state(CHECKPOINT_PATH)
pretrained_layout = panderm.detect_checkpoint_layout(pretrained_state)
assert pretrained_layout == panderm.LAYOUT_DIRECT_BACKBONE
remapped_state = panderm.remap_pretrained_state_dict(pretrained_state, layout=pretrained_layout)
assert "fc_norm.weight" in remapped_state and "fc_norm.bias" in remapped_state
assert not any(key.startswith(("norm.", "head.")) for key in remapped_state)
train_transform = panderm.build_train_transform()
eval_transform = panderm.build_eval_transform()
preflight_model = panderm.build_panderm_classifier(checkpoint_path=CHECKPOINT_PATH, upstream_dir=UPSTREAM_DIR, drop_path=DROP_PATH)
assert preflight_model.pretrained_state_layout == panderm.LAYOUT_DIRECT_BACKBONE
assert preflight_model.head.out_features == 7 and preflight_model.head.in_features == 768
model_details = panderm.model_identity(preflight_model, train_transform=train_transform, eval_transform=eval_transform, checkpoint_sha256=checkpoint_sha256)
dependency_versions = panderm.dependency_versions()
assert type(dependency_versions["torch"]) is str
manifest_sha256 = {split: sha256(SHARED_PROJECT_DIR / "data" / "manifests" / f"{split}.csv") for split in ("train", "val")}
class_mapping_sha256 = sha256(SHARED_PROJECT_DIR / "data" / "manifests" / "class_to_idx.json")
fixed_split_identity = manifest_sha256["train"]
formal_output_identity = f"{sentinel['shared_root_uuid']}:{RUN_VERSION}:formal"
os.environ["DDPM_DERM_DATA_DIR"] = str(SHARED_PROJECT_DIR / "data")
from ddpm_derm import config, manifests, classifier_objective, train_panderm
preflight_args = train_panderm.parse_args([
    "--variant", VARIANT, "--seed", "0", "--epochs", str(VALIDATION_EPOCHS),
    "--batch-size", str(BATCH_SIZE), "--accumulation-steps", str(ACCUMULATION_STEPS),
    "--lr", str(LEARNING_RATE), "--weight-decay", str(WEIGHT_DECAY),
    "--warmup-epochs", str(VALIDATION_EPOCHS), "--layer-decay", str(LAYER_DECAY),
    "--df-target-count", str(DF_TARGET_COUNT), "--checkpoint", str(CHECKPOINT_PATH),
    "--checkpoint-sha256", checkpoint_sha256, "--upstream-dir", str(UPSTREAM_DIR),
    "--upstream-commit", upstream_commit, "--evaluation-scope", "validation_only",
    "--output-dir", "/content/panderm-preflight-output", "--run-version", RUN_VERSION,
    "--shared-root-uuid", sentinel["shared_root_uuid"],
    "--formal-output-identity", formal_output_identity,
    "--fixed-split-identity", fixed_split_identity,
])
preflight_optimizer = panderm.build_optimizer(preflight_model, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, layer_decay=LAYER_DECAY)
assert panderm.verify_optimizer_covers_parameters_once(preflight_optimizer, preflight_model) == sum(1 for parameter in preflight_model.parameters() if parameter.requires_grad)
preflight_num_batches = (panderm_run.EXPECTED_C1_TRAIN_ROWS + BATCH_SIZE - 1) // BATCH_SIZE
preflight_steps_per_epoch = panderm.optimizer_steps_per_epoch(preflight_num_batches, ACCUMULATION_STEPS)
assert preflight_num_batches == 469 and preflight_steps_per_epoch == 58
preflight_schedule = panderm.WarmupCosineSchedule(preflight_optimizer, base_lr=LEARNING_RATE, warmup_epochs=VALIDATION_EPOCHS, epochs=VALIDATION_EPOCHS, steps_per_epoch=preflight_steps_per_epoch)
preflight_scaler = torch.amp.GradScaler("cuda", enabled=True)
production_run_identity = panderm_run.build_run_identity(
    git_commit=commit, seed=0, epochs=VALIDATION_EPOCHS,
    evaluation_scope="validation_only", checkpoint_sha256=checkpoint_sha256,
    model_identity=model_details, manifest_sha256=manifest_sha256,
    fixed_split_identity=fixed_split_identity,
    shared_root_uuid=sentinel["shared_root_uuid"],
    formal_output_identity=formal_output_identity,
    dependency_versions=dependency_versions, warmup_epochs=VALIDATION_EPOCHS,
    drop_path=DROP_PATH, amp_requested=True, amp_effective=True,
    device_type="cuda",
)
panderm_run.require_primitive_identity(production_run_identity)
assert json.loads(json.dumps(production_run_identity, sort_keys=True)) == production_run_identity
print(json.dumps({"factory": phase1_factory.__name__, "layout": pretrained_layout, "dependency_versions": dependency_versions, "run_identity_primitive": True}, indent=2))
del pretrained_state, remapped_state
gc.collect()
print(f"[Phase 1] COMPLETE elapsed={time.monotonic() - phase1_started:.1f}s", flush=True)

## Phase 2 CHECK - dataset-free production checkpoint save and weights-only reopen

In [ ]:
assert not LOCAL_DATA_DIR.exists(), "Phase 2 requires a fresh runtime-local data destination"
phase2_started = time.monotonic()
checkpoint_preflight = train_panderm.checkpoint_serialization_preflight(
    temporary_directory=Path("/content"),
    model=preflight_model,
    optimizer=preflight_optimizer,
    schedule=preflight_schedule,
    scaler=preflight_scaler,
    args=preflight_args,
    run_identity=production_run_identity,
)
assert checkpoint_preflight["weights_only_round_trip"] is True
assert checkpoint_preflight["checkpoint_format"] == CHECKPOINT_FORMAT
assert not list(Path("/content").glob(".panderm-checkpoint-preflight.*.pt"))
del preflight_model, preflight_optimizer, preflight_schedule, preflight_scaler
gc.collect()
print(json.dumps(checkpoint_preflight, indent=2))
print(f"[Phase 2] serialization COMPLETE elapsed={time.monotonic() - phase2_started:.1f}s", flush=True)

import csv, queue, stat, tempfile, threading
PHASE2_EXPECTED_MANIFEST_SHA256 = {
    "train": "eea3fdf281120687b45dfcb5139d927888c6dc84867c67f053c7a743eb02b6fa",
    "val": "22a87a1ab4009c9e87462381f9ef35ad7a5eae7217057049fc24e5531df819f4",
}
PHASE2_EXPECTED_CLASS_MAPPING_SHA256 = "5a034b7dc0c6f44543f558aa589b8e1cba12a05b71a18ff0e2d2029a2ad2e66c"
PHASE2_EXPECTED_CLASS_TO_IDX = {"akiec": 0, "bcc": 1, "bkl": 2, "df": 3, "mel": 4, "nv": 5, "vasc": 6}
def assert_phase2_real_path(path, expected_kind, label):
    path = Path(path)
    assert path.exists(), f"{label} is missing: {path}"
    assert not path.is_symlink(), f"{label} must not be a symlink or reparse point: {path}"
    file_attributes = getattr(path.lstat(), "st_file_attributes", 0)
    reparse_flag = getattr(stat, "FILE_ATTRIBUTE_REPARSE_POINT", 0x400)
    assert not file_attributes & reparse_flag, f"{label} must not be a symlink or reparse point: {path}"
    if expected_kind == "directory":
        assert path.is_dir(), f"{label} is not a directory: {path}"
    elif expected_kind == "file":
        assert path.is_file(), f"{label} is not a file: {path}"
    else:
        raise AssertionError(f"unsupported path kind: {expected_kind}")
    return path.resolve(strict=True)
def verify_test_manifest_data_root(test_data_root):
    test_data_root = Path(test_data_root)
    manifests_root = test_data_root / "manifests"
    assert_phase2_real_path(test_data_root, "directory", "temporary test data root")
    assert_phase2_real_path(manifests_root, "directory", "temporary manifests directory")
    expected_entries = {
        "manifests",
        "manifests/train.csv",
        "manifests/val.csv",
        "manifests/class_to_idx.json",
    }
    entries = list(test_data_root.rglob("*"))
    for path in entries:
        if path.is_dir():
            expected_kind = "directory"
        elif path.is_file():
            expected_kind = "file"
        else:
            raise AssertionError(f"unsupported manifest-only entry: {path}")
        assert_phase2_real_path(path, expected_kind, "manifest-only entry")
    actual_entries = {path.relative_to(test_data_root).as_posix() for path in entries}
    assert actual_entries == expected_entries, f"manifest-only test root drift: {sorted(actual_entries)}"
    actual_files = {path.relative_to(test_data_root).as_posix() for path in entries if path.is_file()}
    assert not (manifests_root / "test.csv").exists(), "test manifest is forbidden in Phase 2"
    assert hashlib.sha256((manifests_root / "train.csv").read_bytes()).hexdigest() == PHASE2_EXPECTED_MANIFEST_SHA256["train"], "authoritative train.csv SHA-256 drift"
    assert hashlib.sha256((manifests_root / "val.csv").read_bytes()).hexdigest() == PHASE2_EXPECTED_MANIFEST_SHA256["val"], "authoritative val.csv SHA-256 drift"
    assert hashlib.sha256((manifests_root / "class_to_idx.json").read_bytes()).hexdigest() == PHASE2_EXPECTED_CLASS_MAPPING_SHA256, "authoritative class_to_idx.json SHA-256 drift"
    with (manifests_root / "train.csv").open("r", encoding="utf-8", newline="") as handle: train_rows = list(csv.DictReader(handle))
    with (manifests_root / "val.csv").open("r", encoding="utf-8", newline="") as handle: val_rows = list(csv.DictReader(handle))
    class_mapping = json.loads((manifests_root / "class_to_idx.json").read_text(encoding="utf-8"))
    assert len(train_rows) == 6995, f"unexpected train row count: {len(train_rows)}"
    assert len(val_rows) == 1510, f"unexpected val row count: {len(val_rows)}"
    assert class_mapping == PHASE2_EXPECTED_CLASS_TO_IDX, f"class mapping drift: {class_mapping!r}"
    return {"files": sorted(actual_files), "train_rows": len(train_rows), "val_rows": len(val_rows)}
def prepare_test_manifest_data_root(source_manifest_root, test_data_root):
    source_manifest_root, test_data_root = Path(source_manifest_root), Path(test_data_root)
    assert_phase2_real_path(source_manifest_root, "directory", "source manifest root")
    assert_phase2_real_path(test_data_root.parent, "directory", "temporary data root parent")
    assert not test_data_root.exists() and not test_data_root.is_symlink(), f"temporary test data root must be fresh: {test_data_root}"
    sources = {
        "train.csv": source_manifest_root / "train.csv",
        "val.csv": source_manifest_root / "val.csv",
        "class_to_idx.json": source_manifest_root / "class_to_idx.json",
    }
    expected_sha256 = {
        "train.csv": PHASE2_EXPECTED_MANIFEST_SHA256["train"],
        "val.csv": PHASE2_EXPECTED_MANIFEST_SHA256["val"],
        "class_to_idx.json": PHASE2_EXPECTED_CLASS_MAPPING_SHA256,
    }
    payloads = {}
    for name, source in sources.items():
        assert_phase2_real_path(source, "file", f"source {name}")
        assert source.parent.samefile(source_manifest_root), f"source {name} escaped the manifest root"
        payloads[name] = source.read_bytes()
        actual_sha256 = hashlib.sha256(payloads[name]).hexdigest()
        assert actual_sha256 == expected_sha256[name], f"authoritative {name} SHA-256 drift"
    manifests_root = test_data_root / "manifests"
    manifests_root.mkdir(parents=True)
    assert_phase2_real_path(test_data_root, "directory", "temporary test data root")
    assert_phase2_real_path(manifests_root, "directory", "temporary manifests directory")
    for name, payload in payloads.items():
        destination = manifests_root / name
        with destination.open("xb") as handle:
            handle.write(payload)
        assert_phase2_real_path(destination, "file", f"temporary {name}")
    return verify_test_manifest_data_root(test_data_root)
def run_stream(command, cwd=CODE_DIR, expect_success=True, process_env=None):
    started = time.monotonic()
    process = subprocess.Popen(command, cwd=cwd, env=process_env or test_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    output_queue = queue.Queue()
    def pump_output():
        for line in process.stdout: output_queue.put(line)
        output_queue.put(None)
    threading.Thread(target=pump_output, daemon=True).start()
    while True:
        try: line = output_queue.get(timeout=60)
        except queue.Empty:
            print(f"[subprocess] heartbeat elapsed={time.monotonic() - started:.1f}s pid={process.pid} alive={process.poll() is None}", flush=True)
            continue
        if line is None: break
        lines.append(line); print(line, end="", flush=True)
    code_returned = process.wait()
    if expect_success and code_returned: raise subprocess.CalledProcessError(code_returned, command)
    if not expect_success and not code_returned: raise AssertionError("command unexpectedly succeeded")
    return time.monotonic() - started, "".join(lines)
assert not Path("/content/panderm-illegal").exists(), "obsolete fixed test output must not exist"
test_temp_root = None
with tempfile.TemporaryDirectory(dir="/content", prefix="panderm-test-manifests-") as temporary:
    test_temp_root = Path(temporary)
    TEST_MANIFEST_DATA_ROOT = test_temp_root / "data"
    TEST_OUTPUT_ROOT = test_temp_root / "outputs"
    test_manifest_report = prepare_test_manifest_data_root(
        SHARED_PROJECT_DIR / "data" / "manifests",
        TEST_MANIFEST_DATA_ROOT,
    )
    assert not LOCAL_DATA_DIR.exists(), "Phase 2 must not create the archive staging destination"
    test_env = os.environ.copy()
    test_env["DDPM_DERM_DATA_DIR"] = str(TEST_MANIFEST_DATA_ROOT)
    test_env["DDPM_DERM_OUTPUTS_DIR"] = str(TEST_OUTPUT_ROOT)
    test_env["PYTHONPATH"] = str(CODE_DIR / "src")
    test_env["PYTHONUNBUFFERED"] = "1"
    test_env["PYTHONDONTWRITEBYTECODE"] = "1"
    _, targeted_output = run_stream([sys.executable, "-B", "-u", "-m", "unittest", "-v", "tests.test_panderm_blockers", "tests.test_panderm_base_c1_finetune", "tests.test_panderm_notebooks", "tests.test_panderm_fresh_runtime"])
    _, suite_output = run_stream([sys.executable, "-B", "-u", "-m", "unittest", "discover", "-s", "tests", "-v"])
    _, runner_smoke_output = run_stream([sys.executable, "-B", "-u", "-m", "unittest", "-v", "tests.test_panderm_blockers.PanDermRunnerMockSmokeTests"])
    targeted_checks = {"targeted_ok": "OK" in targeted_output, "suite_ok": "OK" in suite_output, "runner_smoke_ok": "OK" in runner_smoke_output}
    assert all(targeted_checks.values()), targeted_checks
    illegal = [
        (["--variant", "C4"], "invalid choice"),
        (["--generated-manifest", "/tmp/nope.csv"], "synthetic manifests are rejected"),
        (["--run-version", "v2_other"], "--run-version must be v1_panderm_base_c1_finetune"),
        (["--df-target-count", "586"], "--df-target-count must be 585"),
        (["--accumulation-steps", "0"], "--accumulation-steps must be exactly 8"),
        (["--batch-size", "0"], "--batch-size must be exactly 16"),
        (["--warmup-epochs", "999"], "--warmup-epochs must be exactly 5"),
        (["--evaluation-scope", "full"], "invalid choice"),
        (["--drop-path", "0.3"], "unrecognized arguments"),
        (["--no-amp"], "unrecognized arguments"),
        (["--seed", "1"], "--seed must be exactly 0"),
        (["--epochs", "50"], "--epochs must be exactly 5"),
    ]
    illegal_output_root = TEST_OUTPUT_ROOT / "illegal"
    base_cli = [sys.executable, "-B", "-u", "-m", "ddpm_derm.train_panderm", "--seed", "0", "--epochs", "5", "--warmup-epochs", "5", "--checkpoint", str(CHECKPOINT_PATH), "--upstream-dir", str(UPSTREAM_DIR), "--output-dir", str(TEST_OUTPUT_ROOT / "illegal")]
    _, base_legal_output = run_stream(base_cli + ["--checkpoint-sha256", "0" * 64], expect_success=False)
    assert "SHA-256 mismatch" in base_legal_output
    for extra, expected_error in illegal:
        _, illegal_output = run_stream(base_cli + extra, expect_success=False)
        assert expected_error in illegal_output, {"arguments": extra, "expected_error": expected_error, "output": illegal_output}
    assert not illegal_output_root.exists(), "rejected CLI commands must not create output"
    verify_test_manifest_data_root(TEST_MANIFEST_DATA_ROOT)
    assert not LOCAL_DATA_DIR.exists(), "Phase 2 must remain train/val manifest-only"
    print("CLI illegal combinations rejected:", len(illegal))
assert test_temp_root is not None and not test_temp_root.exists(), "temporary Phase 2 roots were not cleaned"
assert not LOCAL_DATA_DIR.exists(), "Phase 2 must not create the archive staging destination"
assert not Path("/content/panderm-illegal").exists(), "obsolete fixed test output must remain absent"
assert not list(Path("/content").glob("panderm-test-manifests-*")), "Phase 2 temporary residue remains"

## Phase 3 CHECK - manifest-only counts, C1 construction and train/val leakage

In [ ]:
import pandas as pd
phase3_started = time.monotonic()
shared_manifests = SHARED_PROJECT_DIR / "data" / "manifests"
frames = {split: manifests.load_split(split) for split in ("train", "val")}
assert len(frames["train"]) == 6995 and len(frames["val"]) == 1510
assert int((frames["train"]["dx"] == "df").sum()) == 85 and int((frames["val"]["dx"] == "df").sum()) == 14
assert not set(frames["train"]["lesion_id"]) & set(frames["val"]["lesion_id"])
assert not set(frames["train"]["image_id"]) & set(frames["val"]["image_id"])
assert manifest_sha256 == {split: sha256(shared_manifests / f"{split}.csv") for split in ("train", "val")}
c1_frame = manifests.build_classifier_frame("C1", df_target_count=DF_TARGET_COUNT, seed=0)
c1_counts = classifier_objective.ordered_class_counts(c1_frame)
assert len(c1_frame) == panderm_run.EXPECTED_C1_TRAIN_ROWS == 7495
assert c1_counts == panderm_run.EXPECTED_C1_CLASS_COUNTS
assert c1_counts["df"] == DF_TARGET_COUNT
assert "source" not in c1_frame.columns or set(c1_frame["source"].unique()) <= {"real"}
real_df_ids = set(frames["train"].loc[frames["train"]["dx"] == "df", "image_id"].astype(str))
c1_df_ids = set(c1_frame.loc[c1_frame["dx"] == "df", "image_id"].astype(str))
assert c1_df_ids == real_df_ids
assert not c1_df_ids & set(frames["val"]["image_id"].astype(str))
assert not LOCAL_DATA_DIR.exists(), "full local staging must remain unreachable in Phase 3"
print(json.dumps({"train_rows": len(frames["train"]), "val_rows": len(frames["val"]), "c1_rows": len(c1_frame), "c1_counts": c1_counts, "manifest_sha256": manifest_sha256}, indent=2))
print(f"[Phase 3] COMPLETE elapsed={time.monotonic() - phase3_started:.1f}s", flush=True)

## Phase 4 CHECK - fixed four-image train sample, real CUDA update and post-step checkpoint round-trip

In [ ]:
from PIL import Image
phase4_started = time.monotonic()
print("[Phase 4] START four-image CUDA update smoke", flush=True)
assert checkpoint_preflight["weights_only_round_trip"] is True
sample_report = panderm_run.stage_train_smoke_sample(SHARED_PROJECT_DIR / "data", SMOKE_SAMPLE_DIR, count=4)
assert sample_report["source_manifest"] == "manifests/train.csv"
assert sample_report["sample_count"] == 4 and sample_report["test_manifest_read"] is False
sample_paths = [SMOKE_SAMPLE_DIR / Path(*relative.split("/")) for relative in sample_report["relative_paths"]]
model = panderm.build_panderm_classifier(checkpoint_path=CHECKPOINT_PATH, upstream_dir=UPSTREAM_DIR, drop_path=DROP_PATH).cuda()
backbone_count = panderm.assert_full_trainability(model)
total_params, trainable_params = panderm.parameter_counts(model)
assert total_params > trainable_params and model.pos_embed.requires_grad is False
train_panderm.set_seed(0)
sample_tensors = [train_transform(Image.open(path).convert("RGB")) for path in sample_paths]
batch = torch.stack(sample_tensors).cuda()
assert tuple(batch.shape) == (4, 3, 224, 224)
model.train()
logits = model(batch)
assert tuple(logits.shape) == (4, 7) and torch.isfinite(logits).all()
optimizer = panderm.build_optimizer(model, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, layer_decay=LAYER_DECAY)
covered = panderm.verify_optimizer_covers_parameters_once(optimizer, model)
assert covered == sum(1 for parameter in model.parameters() if parameter.requires_grad)
schedule = panderm.WarmupCosineSchedule(optimizer, base_lr=LEARNING_RATE, warmup_epochs=VALIDATION_EPOCHS, epochs=VALIDATION_EPOCHS, steps_per_epoch=preflight_steps_per_epoch)
scaler = torch.amp.GradScaler("cuda", enabled=True)
before_parameters = panderm.snapshot_parameters(model)
targets = torch.tensor([0, 3, 5, 6], device="cuda")
criterion = torch.nn.CrossEntropyLoss()
model.zero_grad(set_to_none=True)
for micro in range(ACCUMULATION_STEPS):
    with torch.autocast(device_type="cuda", enabled=True):
        loss = criterion(model(batch), targets)
    assert torch.isfinite(loss)
    scaler.scale(loss * panderm.accumulation_loss_scale(micro, ACCUMULATION_STEPS, ACCUMULATION_STEPS)).backward()
    print(f"[Phase 4] accumulation micro-step={micro + 1}/{ACCUMULATION_STEPS} loss={float(loss):.4f}", flush=True)
gradient_report = panderm.backbone_gradient_report(model)
assert gradient_report["blocks_with_gradient"] == list(range(12))
assert gradient_report["all_backbone_blocks_have_gradient"]
assert gradient_report["head_parameters_have_gradient"]
assert gradient_report["fixed_backbone_parameter_names"] == ["pos_embed"]
assert model.pos_embed.grad is None
scaler.unscale_(optimizer)
panderm.require_finite_gradients(model.parameters())
scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)
schedule.step()
changed_backbone = panderm.changed_parameter_count(before_parameters, model)
assert changed_backbone > 0
post_step_checkpoint_preflight = train_panderm.checkpoint_serialization_preflight(
    temporary_directory=Path("/content"), model=model, optimizer=optimizer,
    schedule=schedule, scaler=scaler, args=preflight_args,
    run_identity=production_run_identity,
)
assert post_step_checkpoint_preflight["weights_only_round_trip"] is True
backbone_gradient_verified = True
backbone_parameters_updated = True
GPU_SMOKE_COMPLETE = True
del model, optimizer, schedule, scaler, batch, logits, before_parameters, sample_tensors, targets, loss, criterion
gc.collect()
torch.cuda.empty_cache()
shutil.rmtree(SMOKE_SAMPLE_DIR)
gpu_smoke = {
    "official_preprocessing": True,
    "official_checkpoint_loaded": True,
    "cuda_forward": True,
    "backward": True,
    "all_12_blocks_have_gradients": True,
    "head_has_gradients": True,
    "fixed_pos_embed_has_no_gradient": True,
    "optimizer_coverage": True,
    "optimizer_step": True,
    "backbone_updated": True,
    "post_step_checkpoint_round_trip": True,
    "smoke_model_discarded": True,
}
PHASE3_COMPLETE = True
print(json.dumps({"sample": sample_report, "gpu_smoke": gpu_smoke, "changed_backbone_tensors": changed_backbone, "gradient_report": gradient_report}, indent=2))
print(f"[Phase 4] COMPLETE elapsed={time.monotonic() - phase4_started:.1f}s", flush=True)
assert not V1_ROOT.is_symlink(), f"validation version parent must not be a symlink: {V1_ROOT}"
verified_v1_root = panderm_run.ensure_tree(SHARED_RUN_ROOT, V1_ROOT.relative_to(SHARED_RUN_ROOT))
assert verified_v1_root.is_dir() and not verified_v1_root.is_symlink(), f"validation version parent is not a real directory: {verified_v1_root}"
resolved_v1_root = verified_v1_root.resolve(strict=True)
assert resolved_v1_root.is_relative_to(resolved_root), f"validation version parent escaped the shared root: {resolved_v1_root}"
assert resolved_v1_root.parent.samefile(resolved_root), f"validation version parent must be a direct child of the shared root: {resolved_v1_root}"
phase4_version_records = drive_api_list_children(DRIVE_ROOT_ID, RUN_VERSION, panderm_run.DRIVE_FOLDER_MIME_TYPE, drive_id=ROOT_DRIVE_ID, parent_resource_key=DRIVE_ROOT_RESOURCE_KEY)
VALIDATION_VERSION_PROVIDER = panderm_run.require_validation_version_provider_state(
    phase4_version_records,
    expected_parent_id=DRIVE_ROOT_ID,
    run_version=RUN_VERSION,
    topology=VALIDATION_LOCK_STORAGE_TOPOLOGY,
    local_version_exists=True,
)
VALIDATION_VERSION_DRIVE_ID = VALIDATION_VERSION_PROVIDER["id"]
VALIDATION_VERSION_RESOURCE_KEY = VALIDATION_VERSION_PROVIDER.get("resourceKey")
VALIDATION_VERSION_RESOURCE_KEY = VALIDATION_VERSION_RESOURCE_KEY or ""
phase4_active_records = drive_api_list_children(VALIDATION_VERSION_DRIVE_ID, panderm_run.ACTIVE_SESSION_FILENAME, panderm_run.DRIVE_JSON_MIME_TYPE, drive_id=ROOT_DRIVE_ID, parent_resource_key=VALIDATION_VERSION_RESOURCE_KEY)
if phase4_active_records:
    panderm_run.require_active_session_provider_state(phase4_active_records, expected_parent_id=VALIDATION_VERSION_DRIVE_ID, topology=VALIDATION_LOCK_STORAGE_TOPOLOGY, expected_present=True)
else:
    panderm_run.require_active_session_provider_state([], expected_parent_id=VALIDATION_VERSION_DRIVE_ID, topology=VALIDATION_LOCK_STORAGE_TOPOLOGY, expected_present=False)
ACTIVE_SESSION_PATH = panderm_run.active_session_path(SHARED_RUN_ROOT, run_version=RUN_VERSION)
SESSION_HISTORY_DIR = panderm_run.ensure_tree(SHARED_RUN_ROOT, (V1_ROOT / panderm_run.SESSION_HISTORY_DIRECTORY).relative_to(SHARED_RUN_ROOT))
RUN_IDENTITY_SHA256 = panderm_run.canonical_identity_sha256(production_run_identity)
ACTIVE_SESSION = panderm_run.start_sequential_session(ACTIVE_SESSION_PATH, session_id=VALIDATION_SESSION_ID, run_version=RUN_VERSION, git_commit=commit, shared_root_uuid=sentinel["shared_root_uuid"], account_label=ACCOUNT_LABEL, run_identity_sha256=RUN_IDENTITY_SHA256, manual_takeover_confirmed=MANUAL_TAKEOVER_CONFIRMED, history_directory=SESSION_HISTORY_DIR)
# An idempotent manual-takeover retry reuses the replacement session id that
# was already published in the audit, so the effective id always comes from
# the reopened marker rather than from this runtime's generated candidate.
VALIDATION_SESSION_ID = ACTIVE_SESSION["session_id"]
SESSION_RUNTIME_ENV = {"PANDERM_ACTIVE_SESSION_PATH": str(ACTIVE_SESSION_PATH), "PANDERM_ACTIVE_SESSION_ID": VALIDATION_SESSION_ID, "PANDERM_ACTIVE_RUN_VERSION": RUN_VERSION, "PANDERM_ACTIVE_GIT_COMMIT": commit, "PANDERM_ACTIVE_SHARED_ROOT_UUID": sentinel["shared_root_uuid"], "PANDERM_ACTIVE_RUN_IDENTITY_SHA256": RUN_IDENTITY_SHA256}
SESSION_GUARD = panderm_run.SequentialSessionWriteGuard.from_environment(SESSION_RUNTIME_ENV)
SESSION_GUARD.bind_run_identity(production_run_identity)
def require_active_session(phase):
    return SESSION_GUARD.require(phase)
print("[sequential-session] active", VALIDATION_SESSION_ID, "checkpoint_cadence=every_epoch max_quota_loss=one_incomplete_epoch", flush=True)

## Phase 5 RUN - verified immutable train/val-only archive staging

In [ ]:
assert checkpoint_preflight["weights_only_round_trip"] is True
assert post_step_checkpoint_preflight["weights_only_round_trip"] is True
assert PHASE3_COMPLETE is True
APPROVED_CONTENT_IDENTITY = panderm_run.require_approved_content_identity(
    panderm_run.EXPECTED_VALIDATION_CONTENT_IDENTITY_SHA256
)
assert APPROVED_CONTENT_IDENTITY != panderm_run.VALIDATION_CONTENT_IDENTITY_PLACEHOLDER
assert len(APPROVED_CONTENT_IDENTITY) == 64
require_active_session("Phase 5 archive staging start")
require_active_session("Phase 5 data cache parent preparation")
data_cache_parent = panderm_run.ensure_tree(SHARED_RUN_ROOT, Path("data_cache"))
assert DATA_CACHE_DIRECTORY.parent.resolve(strict=True) == data_cache_parent.resolve(strict=True)
def stage_from_verified_archive():
    if not DATA_CACHE_DIRECTORY.exists():
        panderm_run.build_validation_archive_cache(
            SHARED_PROJECT_DIR / "data", DATA_CACHE_DIRECTORY, Path("/content"),
            expected_file_content_identity_sha256=APPROVED_CONTENT_IDENTITY,
            source_fixed_split_identity=fixed_split_identity,
            write_guard=require_active_session,
        )
    return panderm_run.reuse_validation_archive_cache(
        DATA_CACHE_DIRECTORY, Path("/content"), LOCAL_DATA_DIR,
        expected_file_content_identity_sha256=APPROVED_CONTENT_IDENTITY,
        expected_fixed_split_identity=fixed_split_identity,
        expected_manifest_sha256=manifest_sha256,
        expected_class_mapping_sha256=class_mapping_sha256,
        write_guard=require_active_session,
    )
staging_report = panderm_run.stage_after_validation_preflights(
    checkpoint_preflight=checkpoint_preflight,
    gpu_smoke=gpu_smoke,
    staging=stage_from_verified_archive,
)
assert staging_report["archive_files_copied"] == 1
assert staging_report["images_staged"] == 8505
assert staging_report["test_manifest_present"] is False
assert staging_report["whole_data_copy_used"] is False
assert staging_report["approved_file_content_identity_sha256"] == APPROVED_CONTENT_IDENTITY
for required_manifest in ("train.csv", "val.csv", "class_to_idx.json"):
    assert (LOCAL_DATA_DIR / "manifests" / required_manifest).is_file()
assert not (LOCAL_DATA_DIR / "manifests" / "test.csv").exists()
archive_identity = panderm_run.validate_validation_archive_cache(
    DATA_CACHE_DIRECTORY,
    expected_file_content_identity_sha256=APPROVED_CONTENT_IDENTITY,
    expected_fixed_split_identity=fixed_split_identity,
    expected_manifest_sha256=manifest_sha256,
    expected_class_mapping_sha256=class_mapping_sha256,
    write_guard=require_active_session,
)
training_env = os.environ.copy()
training_env["DDPM_DERM_DATA_DIR"] = str(LOCAL_DATA_DIR)
training_env["PYTHONPATH"] = str(CODE_DIR / "src")
training_env["PYTHONUNBUFFERED"] = "1"
training_env["PYTHONDONTWRITEBYTECODE"] = "1"
training_env.update(SESSION_RUNTIME_ENV)
assert archive_identity["file_content_identity_sha256"] == APPROVED_CONTENT_IDENTITY
require_active_session("Phase 5 archive staging completion")
print(json.dumps({"staging_report": staging_report, "archive_identity": archive_identity, "approved_content_identity": APPROVED_CONTENT_IDENTITY}, indent=2))

## Phase 6 RUN - fresh seed-0 five-epoch validation-only subprocess

In [ ]:
import copy
from datetime import datetime, timezone
require_active_session("Phase 6 attempt preflight")
panderm_run.ensure_tree(SHARED_RUN_ROOT, V1_ROOT.relative_to(SHARED_RUN_ROOT))
require_active_session("Phase 6 validation runs root preparation")
panderm_run.ensure_tree(SHARED_RUN_ROOT, VALIDATION_RUNS_ROOT.relative_to(SHARED_RUN_ROOT))
PHASE6_ATTEMPT_IDENTITY_SCHEMA_VERSION = 3
PHASE6_RESUMABLE_ARTIFACT_NAMES = ("last.pt", "last.pt.integrity.json", "best.pt", "best.pt.integrity.json")
def phase6_attempt_identity(validation_id, run_version, shared_root_uuid, run_identity):
    """Immutable, cross-session attempt identity.

    Session id, account label and hostname are deliberately absent so that
    account B resumes exactly the attempt account A started.
    """
    return {"schema_version": PHASE6_ATTEMPT_IDENTITY_SCHEMA_VERSION, "validation_id": str(validation_id), "run_version": str(run_version), "shared_root_uuid": str(shared_root_uuid), "run_identity": run_identity}
def phase6_session_metadata(session_id, account_label, hostname, started_utc):
    """Per-session operational metadata, recorded beside the attempt identity."""
    return {"schema_version": 1, "session_id": str(session_id), "account_label": str(account_label), "hostname": str(hostname), "started_utc": str(started_utc)}
def phase6_classify_existing_artifacts(checkpoint_dir, result_path, run_identity, epochs):
    """Classify existing durable artifacts as fresh, resume or verified complete.

    Only corrupt bytes, an invalid sidecar or immutable identity drift may
    reject. A different session id, account label or hostname never does, and
    an existing complete pair is verified and skipped instead of overwritten.
    """
    checkpoint_dir, result_path = Path(checkpoint_dir), Path(result_path)
    last_path, best_path = checkpoint_dir / "last.pt", checkpoint_dir / "best.pt"
    present = [name for name in PHASE6_RESUMABLE_ARTIFACT_NAMES if (checkpoint_dir / name).exists()]
    if not present and not result_path.exists():
        return {"mode": "fresh", "present": present, "epoch": None, "global_step": None}
    assert last_path.exists(), f"durable artifacts exist without last.pt; inspect manually: {present}"
    last_record = train_panderm.checkpoint_integrity_record(last_path, expected_identity=run_identity)
    if result_path.exists():
        completed = json.loads(result_path.read_text(encoding="utf-8"))
        panderm_run.require_matching_identity(completed["run_identity"], run_identity)
        completed_best, completed_last = train_panderm.load_completed_checkpoint_pair_safe(best_path=best_path, last_path=last_path, result=completed, model=None, expected_identity=run_identity, map_location="cpu")
        panderm_run.require_completed_artifact_identities(expected=run_identity, result=completed, best_checkpoint=completed_best, last_checkpoint=completed_last)
        assert last_record["epoch"] == int(epochs), f"completed result with an unfinished last.pt epoch: {last_record['epoch']}"
        return {"mode": "complete", "present": present, "epoch": last_record["epoch"], "global_step": last_record["global_step"]}
    assert last_record["epoch"] < int(epochs), f"last.pt reached the fixed budget without a published result: {last_record['epoch']}"
    return {"mode": "resume", "present": present, "epoch": last_record["epoch"], "global_step": last_record["global_step"]}
existing_attempts = sorted(path for path in VALIDATION_RUNS_ROOT.iterdir() if path.is_dir())
assert len(existing_attempts) <= 1, f"only one validation attempt directory is permitted: {existing_attempts}"
if existing_attempts:
    VALIDATION_DIR = existing_attempts[0]
    validation_id = VALIDATION_DIR.name
else:
    validation_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    require_active_session("Phase 6 attempt directory preparation")
    VALIDATION_DIR = panderm_run.ensure_tree(SHARED_RUN_ROOT, (VALIDATION_RUNS_ROOT / validation_id).relative_to(SHARED_RUN_ROOT))
ATTEMPT_IDENTITY = phase6_attempt_identity(validation_id, RUN_VERSION, sentinel["shared_root_uuid"], production_run_identity)
attempt_identity_path = VALIDATION_DIR / "attempt_identity.json"
if attempt_identity_path.exists():
    assert json.loads(attempt_identity_path.read_text(encoding="utf-8")) == ATTEMPT_IDENTITY, "immutable attempt run identity drift"
else:
    panderm_run.write_json_atomic(attempt_identity_path, ATTEMPT_IDENTITY, write_guard=require_active_session)
require_active_session("Phase 6 attempt session metadata")
ATTEMPT_SESSIONS_DIR = panderm_run.ensure_tree(SHARED_RUN_ROOT, (VALIDATION_DIR / "attempt_sessions").relative_to(SHARED_RUN_ROOT))
ATTEMPT_SESSION_METADATA = phase6_session_metadata(VALIDATION_SESSION_ID, ACCOUNT_LABEL, ACTIVE_SESSION["hostname"], ACTIVE_SESSION["started_utc"])
attempt_session_path = ATTEMPT_SESSIONS_DIR / f"{VALIDATION_SESSION_ID}.json"
if not attempt_session_path.exists():
    panderm_run.write_json_atomic(attempt_session_path, ATTEMPT_SESSION_METADATA, write_guard=require_active_session)
require_active_session("Phase 6 gate directory preparation")
GATE_ROOT = panderm_run.ensure_tree(SHARED_RUN_ROOT, (VALIDATION_DIR / "non_collapse_gate").relative_to(SHARED_RUN_ROOT))
gate_command = [sys.executable, "-B", "-u", "-m", "ddpm_derm.train_panderm", "--variant", VARIANT, "--seed", "0", "--epochs", str(VALIDATION_EPOCHS), "--batch-size", str(BATCH_SIZE), "--accumulation-steps", str(ACCUMULATION_STEPS), "--lr", str(LEARNING_RATE), "--weight-decay", str(WEIGHT_DECAY), "--warmup-epochs", str(VALIDATION_EPOCHS), "--layer-decay", str(LAYER_DECAY), "--df-target-count", str(DF_TARGET_COUNT), "--checkpoint", str(CHECKPOINT_PATH), "--checkpoint-sha256", checkpoint_sha256, "--upstream-dir", str(UPSTREAM_DIR), "--upstream-commit", upstream_commit, "--evaluation-scope", "validation_only", "--output-dir", str(GATE_ROOT), "--run-version", RUN_VERSION, "--shared-root-uuid", sentinel["shared_root_uuid"], "--formal-output-identity", formal_output_identity, "--fixed-split-identity", fixed_split_identity, "--resume"]
checkpoint_dir = GATE_ROOT / "checkpoints" / ARCH / f"{VARIANT}_seed0"
result_path = GATE_ROOT / "results" / ARCH / f"results_{VARIANT}_seed0.json"
RESUME_STATE = phase6_classify_existing_artifacts(checkpoint_dir, result_path, production_run_identity, VALIDATION_EPOCHS)
print(f"[Phase 6] durable artifacts mode={RESUME_STATE['mode']} epoch={RESUME_STATE['epoch']} global_step={RESUME_STATE['global_step']} present={RESUME_STATE['present']}", flush=True)
gate_seconds, gate_output = run_stream(gate_command, process_env=training_env)
if RESUME_STATE["mode"] == "fresh":
    assert "[start] --resume set but no checkpoint yet -> fresh run from epoch 1" in gate_output
elif RESUME_STATE["mode"] == "resume":
    assert "[resume] found last.pt" in gate_output
else:
    assert "[skip] already completed all" in gate_output
assert "[test]" not in gate_output
assert RESUME_STATE["mode"] == "complete" or "checkpoint_saved=last.pt" in gate_output
result = json.loads(result_path.read_text(encoding="utf-8"))
panderm_run.require_matching_identity(result["run_identity"], production_run_identity)
assert result["evaluation_scope"] == "validation_only" and result["test_metrics"] is None
assert result["data_counts"] == {"train": 7495, "val": 1510, "test": None}
assert len(result["history"]) == VALIDATION_EPOCHS
best_path, last_path = checkpoint_dir / "best.pt", checkpoint_dir / "last.pt"
assert best_path.is_file() and last_path.is_file()
for path in (best_path, last_path):
    reopened = train_panderm.load_checkpoint_safe(path, map_location="cpu", expected_identity=result["run_identity"], expected_result_checkpoint=result["checkpoint_integrity"][path.name])
    assert reopened["checkpoint_format"] == CHECKPOINT_FORMAT
    assert "model_state_dict" in reopened and "head_state_dict" not in reopened
    assert set(reopened) >= {"optimizer_state_dict", "scheduler_state_dict", "scaler_state_dict", "rng_state", "run_identity"}
    assert reopened["run_identity"] == result["run_identity"]
    panderm_run.require_matching_identity(reopened["run_identity"], result["run_identity"])
assert train_panderm.load_checkpoint_safe(last_path, map_location="cpu", expected_identity=result["run_identity"], expected_result_checkpoint=result["checkpoint_integrity"]["last.pt"])["epoch"] == VALIDATION_EPOCHS
best_checkpoint, last_checkpoint = train_panderm.load_completed_checkpoint_pair_safe(best_path=best_path, last_path=last_path, result=result, model=None, expected_identity=result["run_identity"], map_location="cpu")
panderm_run.require_completed_artifact_identities(expected=result["run_identity"], result=result, best_checkpoint=best_checkpoint, last_checkpoint=last_checkpoint)
before_state = (sha256(last_path), last_path.stat().st_mtime_ns)
_, resumed_output = run_stream(gate_command, process_env=training_env)
assert "[skip] already completed all" in resumed_output
for index, replacement in (("--layer-decay", "0.75"), ("--lr", "1e-4"), ("--weight-decay", "0.01"), ("--accumulation-steps", "4"), ("--batch-size", "8")):
    mismatch = gate_command.copy()
    mismatch[mismatch.index(index) + 1] = replacement
    run_stream(mismatch, expect_success=False, process_env=training_env)
saved_identity = last_checkpoint["run_identity"]
for key, value in (("checkpoint_sha256", "0" * 64), ("upstream_commit", "1" * 40), ("claim_boundary", "confirmed"), ("evaluation_scope", "full")):
    drifted = copy.deepcopy(saved_identity)
    drifted[key] = value
    try: panderm_run.require_matching_identity(saved_identity, drifted); raise AssertionError(f"identity drift accepted: {key}")
    except ValueError: pass
truncated = {key: value for key, value in saved_identity.items() if key != "objective"}
try: panderm_run.require_expected_identity_complete(truncated); raise AssertionError("truncated expectation accepted")
except ValueError: pass
assert (sha256(last_path), last_path.stat().st_mtime_ns) == before_state, "a rejected resume must not mutate the checkpoint"
child_code = "import subprocess,sys; subprocess.run(sys.argv[1:], check=True)"
subprocess.run([sys.executable, "-B", "-u", "-c", child_code] + gate_command, cwd=CODE_DIR, env=training_env, check=True)
resume_checks = {"resume": True, "initial_gate_call_uses_resume": True, "resume_mode": RESUME_STATE["mode"], "mismatches_rejected_without_mutation": True, "child_process_drive_only_restore": True}
print(json.dumps({"validation_id": validation_id, "gate_seconds": gate_seconds, "best_val_df_f1": result["best_val_df_f1"], "attempt_identity": ATTEMPT_IDENTITY["schema_version"], "resume_state": RESUME_STATE, "resume_checks": resume_checks}, indent=2))

## Phase 7 REVIEW - result, best/last reopen, resume, mismatch rejection and non-collapse gate

In [ ]:
predicted = torch.tensor(result["validation_metrics"]["confusion_matrix"]).sum(dim=0)
prediction_counts = {name: int(predicted[index]) for index, name in enumerate(config.CLASS_NAMES)}
identity_complete = set(result["run_identity"]) == set(panderm_run.IMMUTABLE_IDENTITY_KEYS)
trainer_source = (CODE_DIR / "src" / "ddpm_derm" / "train_panderm.py").read_text(encoding="utf-8")
validation_notebook = json.loads((CODE_DIR / "notebooks" / "colab_panderm_base_c1_finetune_validation.ipynb").read_text(encoding="utf-8"))
validation_code = "\n".join("".join(cell.get("source", [])) for cell in validation_notebook["cells"] if cell["cell_type"] == "code")
panderm_run.validate_validation_notebook_source(validation_code)
formal_notebook = json.loads((CODE_DIR / "notebooks" / "colab_panderm_base_c1_finetune_classifier.ipynb").read_text(encoding="utf-8"))
formal_code = "\n".join("".join(cell.get("source", [])) for cell in formal_notebook["cells"] if cell["cell_type"] == "code")
source_guards = {
    "trainer_has_no_test_split_load": 'load_split("test")' not in trainer_source,
    "trainer_has_no_full_scope": 'evaluation_scope == "full"' not in trainer_source,
    "validation_notebook_uses_archive_build": "panderm_run.build_validation_archive_cache" in validation_code,
    "validation_notebook_uses_single_tar_reuse": "panderm_run.reuse_validation_archive_cache" in validation_code,
    "validation_notebook_has_no_legacy_full_copy": "panderm_run.stage_validation_data" not in validation_code,
    "validation_notebook_copy_ast_safe": True,
    "formal_notebook_has_no_test_split_load": 'load_split("test")' not in formal_code,
    "formal_notebook_has_no_training_entrypoint": "ddpm_derm.train_panderm" not in formal_code,
}
test_access_probes = {}
for scenario in ("before_validation_match", "before_three_seed_completion", "forged_validation_pass"):
    try:
        panderm_run.require_provenance_clearance(upstream_commit=upstream_commit, checkpoint_sha256=checkpoint_sha256, purpose=panderm_run.TEST_ACCESS)
    except ValueError as error:
        test_access_probes[scenario] = str(error) == panderm_run.PROHIBITED_FORMAL_TEST_REASON
    else:
        test_access_probes[scenario] = False
no_test_access = all(source_guards.values()) and all(test_access_probes.values()) and result["test_metrics"] is None and result["data_counts"]["test"] is None
checks = panderm_run.evaluate_non_collapse_gate(result=result, prediction_counts=prediction_counts, backbone_gradient_verified=backbone_gradient_verified, backbone_parameters_updated=backbone_parameters_updated, identity_complete=identity_complete, no_test_access=no_test_access, provenance_allows_next_stage=bool(provenance_clearance))
assert set(checks) == set(panderm_run.NON_COLLAPSE_CHECK_KEYS)
gate_metrics = {"best_validation_df_f1": result["best_val_df_f1"], "best_epoch": max(result["history"], key=lambda item: item["val_df_f1"])["epoch"], "history": result["history"], "prediction_counts": prediction_counts, "checks": checks, "test_access_probes": test_access_probes, "source_guards": source_guards, "elapsed_seconds": gate_seconds, "changed_backbone_tensors": changed_backbone, "gradient_report": gradient_report}
gate_failures = [key for key, value in checks.items() if not value]
if gate_failures:
    failure_guard_after = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
    assert failure_guard_after == before_guard, "pre-existing read-only guard files changed"
    failure_record = {"validation_status": "VALIDATION FAILED", "formal_training_started": False, "formal_training_allowed": False, "test_access_allowed": False, "run_version": RUN_VERSION, "git_commit": commit, "upstream_commit": upstream_commit, "checkpoint_sha256": checkpoint_sha256, "arch": ARCH, "variant": VARIANT, "gate_failures": gate_failures, "non_collapse_gate": gate_metrics, "model_identity": model_details, "fixed_split_identity": fixed_split_identity, "manifest_sha256": manifest_sha256, "shared_root_uuid": sentinel["shared_root_uuid"], "durable_root_provider_identity": DURABLE_ROOT_PROVIDER_IDENTITY, "active_session": ACTIVE_SESSION, "provenance": provenance, "claim_boundary": panderm_run.CLAIM_BOUNDARY, "evaluation_scope": "validation_only", "test_metrics": None, "validation_artifact_directory": str(VALIDATION_DIR)}
    panderm_run.write_json_atomic(VALIDATION_DIR / "validation_failure.json", failure_record, write_guard=require_active_session)
    panderm_run.write_json_atomic(LATEST_FAILURE_RECORD, failure_record, write_guard=require_active_session)
    assert not VALIDATION_RECORD.exists()
    print("VALIDATION FAILED")
    print("formal_training_allowed=false")
    print("test_access_allowed=false")
    print("[sequential-session] failure retained the active marker for explicit review or manual takeover", flush=True)
    raise RuntimeError(gate_failures)
print(json.dumps(checks, indent=2))

### Phase 7 REVIEW - immutable validation record and formal/test hard stop

In [ ]:
guard_after = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
assert guard_after == before_guard, "pre-existing read-only guard files changed"
panderm_run.require_no_deployment_contamination(CODE_DIR)
record = {"validation_status": "VALIDATION PASSED", "formal_training_started": False, "formal_training_allowed": False, "test_access_allowed": False, "run_version": RUN_VERSION, "arch": ARCH, "variant": VARIANT, "git_commit": commit, "upstream_repo": PANDERM_UPSTREAM_REPO, "upstream_commit": upstream_commit, "checkpoint_filename": PANDERM_CHECKPOINT_FILENAME, "checkpoint_source_url": panderm_run.CHECKPOINT_SOURCE_URL, "checkpoint_sha256": checkpoint_sha256, "checkpoint_sha256_provenance": panderm_run.CHECKPOINT_SHA256_PROVENANCE, "checkpoint_bytes": CHECKPOINT_PATH.stat().st_size, "checkpoint_format": CHECKPOINT_FORMAT, "model_identity": model_details, "dependency_versions": dependency_versions, "fixed_split_identity": fixed_split_identity, "manifest_sha256": manifest_sha256, "data_cache_identity": archive_identity, "c1_construction": {"strategy": "duplicate_real_train_df", "df_target_count": DF_TARGET_COUNT, "synthetic_images_used": False, "class_counts": c1_counts, "train_rows": len(c1_frame)}, "objective": result["run_identity"]["objective"], "optimization": result["run_identity"]["optimization"], "shared_root_uuid": sentinel["shared_root_uuid"], "shared_root_identity": sentinel, "durable_root_provider_identity": DURABLE_ROOT_PROVIDER_IDENTITY, "active_session": ACTIVE_SESSION, "formal_output_identity": formal_output_identity, "evaluation_scope": "validation_only", "non_collapse_gate": gate_metrics, "resume_mismatch_checks": resume_checks, "targeted_check_results": targeted_checks, "drive_probes": drive_probe, "provenance": provenance, "license_review": provenance["license_review"], "contamination_review": provenance["contamination_review"], "claim_boundary": panderm_run.CLAIM_BOUNDARY, "results_grade": "exploratory", "deployment_allowed": False, "attribution": panderm_run.ATTRIBUTION, "validation_artifact_directory": str(VALIDATION_DIR), "existing_artifacts_unchanged": True, "gh_token_present": GH_TOKEN_PRESENT, "test_metrics": None}
panderm_run.write_json_atomic(VALIDATION_DIR / "validation_record.json", record, write_guard=require_active_session)
panderm_run.write_json_atomic(VALIDATION_RECORD, record, write_guard=require_active_session)
assert not FORMAL_ROOT.exists(), "PanDerm v1 must never create formal artifacts"
assert json.loads(VALIDATION_RECORD.read_text(encoding="utf-8")) == record
# Result, checkpoints and validation record are verified above; only now is
# this sequential session marked complete for the next account's handoff.
COMPLETED_SESSION_RECORD = panderm_run.complete_sequential_session(ACTIVE_SESSION_PATH, session_id=VALIDATION_SESSION_ID, history_directory=SESSION_HISTORY_DIR, checkpoint_integrity=result["checkpoint_integrity"], result_identity={"epoch": result["epoch"], "global_step": result["global_step"], "run_identity_sha256": RUN_IDENTITY_SHA256})
print("[sequential-session] completed by", VALIDATION_SESSION_ID, flush=True)
print(json.dumps(record, indent=2))
print("VALIDATION PASSED")
print("formal_training_allowed=false")
print("test_access_allowed=false")
print("run_version=v1_panderm_base_c1_finetune")
print("claim_boundary=suggestive_exploratory_only")